# FoodVision Lab: run the experimental pipeline on a GPU
1. *Runtime > Change runtime type > T4 GPU*.  2. Set `REPO_URL` and run all cells.
3. Unzip the downloaded `artifacts.zip` into your repository, run `python -m foodvision.reports` to refresh the
   README's results table, then commit and push.

In [ ]:
REPO_URL = "https://github.com/AnushGoel/foodvision-lab.git"  # <- your repository

import os, sys
if not os.path.isdir("/content/foodvision-lab"):
    !git clone -q {REPO_URL} /content/foodvision-lab
%cd /content/foodvision-lab
!pip -q install -r requirements-train.txt
sys.path.insert(0, os.path.abspath("src"))   # import the package directly (no kernel restart needed)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime > Change runtime type > T4 GPU")

## Google Drive backup
Saves results to Google Drive **as they are produced**, so a disconnect or a blocked download loses nothing. If Colab
disconnects, run all cells again: finished stages are read back from Drive and skipped. Colab asks for permission to
use Drive (click *Connect to Google Drive* and *Allow*), so stay for this step. Set `SAVE_TO_DRIVE = False` to skip.

In [ ]:
SAVE_TO_DRIVE = True
BACKUP = "/content/drive/MyDrive/foodvision-lab-results"

import os, shutil
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for folder in ("artifacts", "runs"):
        target = os.path.join(BACKUP, folder)
        if not os.path.islink(folder):
            if os.path.isdir(folder):
                shutil.copytree(folder, target, dirs_exist_ok=True)  # keeps earlier results already on Drive
                shutil.rmtree(folder)
            os.makedirs(target, exist_ok=True)
            os.symlink(target, folder)
    print("Results are saved to Google Drive as they are produced:", BACKUP)
else:
    print("Drive backup is off: results stay on this Colab machine until you download them.")

## 1. Quick check (about 5 minutes)
Runs the whole pipeline on one small model with tiny budgets, so any problem shows up here instead of after the full
run. It also downloads, once, the public non-food photos (about 1.1 GB) used by the not-food check. Its results are
deleted afterwards.

In [ ]:
import shutil
from foodvision.pipeline import main

main(["--config", "configs/smoke.yaml"])
for folder in ("smoke_artifacts", "smoke_runs", "smoke_checkpoints"):
    shutil.rmtree(folder, ignore_errors=True)
print("Check passed: the full run can start.")

## 2. Full run (about 20 minutes on a T4)
Data audit, the nine assignment runs, the research protocol (three seeds, validation-based selection, one-time test
evaluation, calibration, not-food check, BatchNorm ablation, Grad-CAM gallery), the analysis of the selected model
(robustness test, feature map, data-efficiency curve) and a latency benchmark. A rising chime plays when it finishes
and a falling tone if it fails, so you can leave the tab. If Colab disconnects, run all cells again: finished stages
are skipped.

In [ ]:
import numpy as np
from IPython.display import Audio, display


def chime(ok: bool = True) -> None:
    rate, t = 22050, np.linspace(0, 0.3, int(22050 * 0.3), endpoint=False)
    notes = (660, 880, 1320) if ok else (440, 330, 220)
    display(Audio(np.concatenate([np.sin(2 * np.pi * f * t) * np.exp(-6 * t) for f in notes]), rate=rate, autoplay=True))


try:
    main(["--config", "configs/default.yaml"])
    chime(ok=True)
except Exception:
    chime(ok=False)
    raise

## 3. TensorBoard (optional)
`runs/assignment` holds the nine assignment runs; `runs/research` the seeded validation runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 4. Results and download
Shows the summary figure (ready for your README or a LinkedIn post; it is also saved as
`artifacts/figures/summary.png`) and the generated experiments report, then downloads `artifacts.zip` (upload it to Claude, or commit it to the
repository) and `tensorboard_runs.zip` (keep it for yourself). If your browser asks about downloading several files,
click *Allow*. With the Drive backup on, both zips are also saved in your Drive folder.

In [ ]:
from IPython.display import Markdown, display
for name, archive in (("artifacts", "artifacts"), ("runs", "tensorboard_runs")):
    real = os.path.realpath(name)  # works whether the folder lives here or on Drive
    shutil.make_archive(archive, "zip", root_dir=os.path.dirname(real), base_dir=os.path.basename(real))
from IPython.display import Image as ShowImage
if os.path.exists("artifacts/figures/summary.png"):
    display(ShowImage("artifacts/figures/summary.png"))
display(Markdown(open("artifacts/reports/EXPERIMENTS.md").read()))
if SAVE_TO_DRIVE:
    for archive in ("artifacts.zip", "tensorboard_runs.zip"):
        shutil.copy(archive, BACKUP)
    print("Zips also saved to Google Drive:", BACKUP)
from google.colab import files
files.download("artifacts.zip")
files.download("tensorboard_runs.zip")

## 5. Try your own photos (optional)
Upload any photos: pizza, steak, sushi, or something else entirely, like your pet. This runs the same code as the
website, so it also checks the model before you deploy. Click *Choose Files*, or press the stop button to skip.

In [ ]:
import os, sys

MODEL = "/content/foodvision-lab/artifacts/model/foodvision.pt"
if not os.path.exists(MODEL):
    print("This Colab session was reset (that happens after a disconnect), so the project isn't loaded.\n"
          "Run the first cell (Set up FoodVision Lab) and the Google Drive cell, then this one again. Your\n"
          "trained model is saved in Google Drive, so nothing needs retraining. If the Drive backup was off,\n"
          "upload the artifacts.zip you downloaded and unzip it into /content/foodvision-lab instead.")
else:
    os.chdir("/content/foodvision-lab")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
    import torch
    from google.colab import files
    from IPython.display import display
    from PIL import Image
    from foodvision.inference import InvalidImageError, Predictor, load_image

    predictor = Predictor(MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
    for name, data in files.upload().items():
        try:
            image = load_image(data)
        except InvalidImageError as error:
            print(f"{name}: {error}")
            continue
        p = predictor.predict(image, abstain_below=0.6)
        view, heat = predictor.explain(image)
        if p.in_distribution is False:
            verdict = f"not pizza, steak or sushi? (closest class: {p.label}, {p.confidence:.0%})"
        elif p.abstained:
            verdict = f"not sure (top class {p.label}, {p.confidence:.0%})"
        else:
            verdict = f"{p.label}, {p.confidence:.0%} confidence"
        print(f"\n{name}: {verdict}")
        print("  " + ", ".join(f"{c} {q:.1%}" for c, q in zip(p.classes, p.probabilities)))
        if p.ood_score is not None:
            print(f"  not-food score {p.ood_score:.2f} (threshold {predictor.ood['threshold']:.2f})")
        side = Image.new("RGB", (view.width * 2, view.height), "white")
        side.paste(view, (0, 0))
        side.paste(heat, (view.width, 0))
        display(side.resize((520, 260)))